# 作业 02：把多个输入接入上周的训练模板

作业 01 比较了同一个样本的两种表示：两个标量输入张量，或一个包含两个数的输入张量。本题把多个样本分别放入 x1、x2，练习补全训练代码。

请参考上周模板，让同一个预测函数同时使用 x1 和 x2。所有样本共享 w1、w2、b；每次预测都需要两个输入。下面提供函数和数据，参数创建、损失、梯度与更新由你完成。

本文件可以独立运行，不依赖作业 01 的内核变量。


In [1]:
import torch

def predict(x1, x2, w1, w2, b):
    return w1 * x1 + w2 * x2 + b


## 训练数据：每一行是同一个样本的两个输入和一个目标

下面 x1、x2、y 的 shape 都为 `(6,)`，表示 6 个样本。`predict(x1, x2, ...)` 同时使用两个输入张量，返回 6 个预测值。

| 样本 i | x1[i] | x2[i] | y[i] |
|---|---|---|---|
| 0 | -2 | 1 | -6 |
| 1 | -1 | 0 | -1 |
| 2 | 0 | 2 | -5 |
| 3 | 1 | -1 | 6 |
| 4 | 2 | 3 | -4 |
| 5 | 3 | 1 | 4 |

训练时不要修改输入和目标，也不要直接填写猜测的参数答案。


In [ ]:
x1 = torch.tensor([-2., -1., 0., 1., 2., 3.])
x2 = torch.tensor([1., 0., 2., -1., 3., 1.])
y = torch.tensor([-6., -1., -5., 6., -4., 4.])
print('x1:', x1, 'shape:', tuple(x1.shape))
print('x2:', x2, 'shape:', tuple(x2.shape))
print('y: ', y, 'shape:', tuple(y.shape))


## 你的训练区：修改上周模板

创建三个浮点标量参数 w1、w2、b，都启用梯度，并一起交给优化器。每轮调用给定的 predict 函数，同时传入 x1、x2，得到 shape 为 `(6,)` 的 y_pred。

使用绝对值损失：先得到 `(6,)` 的逐样本误差，再用 `.mean()` 得到 shape 为 `()` 的平均损失。只对这个平均损失 backward 一次，然后更新全部参数。

可以从三个参数都为 0、2000 轮、学习率 `0.1 / (epoch ** 0.5)` 开始。这里预测公式是线性的，零初始化与上周的平方参数例子不同。

补完后删除 NotImplementedError。


In [5]:
import torch

def predict(x1, x2, w1, w2, b):
    return w1 * x1 + w2 * x2 + b

x1 = torch.tensor([-2., -1., 0., 1., 2., 3.])
x2 = torch.tensor([1., 0., 2., -1., 3., 1.])
y  = torch.tensor([-6., -1., -5., 6., -4., 4.])

print('x1:', x1, 'shape:', tuple(x1.shape))
print('x2:', x2, 'shape:', tuple(x2.shape))
print('y: ', y,  'shape:', tuple(y.shape))

w1 = torch.tensor(0., requires_grad=True)
w2 = torch.tensor(0., requires_grad=True)
b  = torch.tensor(0., requires_grad=True)

optimizer = torch.optim.SGD([w1, w2, b], lr=0.1)

for epoch in range(1, 2001):
    for pg in optimizer.param_groups:
        pg['lr'] = 0.1 / (epoch ** 0.5)
    optimizer.zero_grad()

    y_pred = predict(x1, x2, w1, w2, b)          

    loss_per_sample = torch.abs(y - y_pred)       
    loss = loss_per_sample.mean()                   

    loss.backward()
    optimizer.step()

with torch.no_grad():
    y_pred_final = predict(x1, x2, w1, w2, b)
    loss_per_sample_final = torch.abs(y - y_pred_final)
    loss_final = loss_per_sample_final.mean()

    print(f'\n===== 最终结果 =====')
    print(f'w1 = {w1.item():.6f}')
    print(f'w2 = {w2.item():.6f}')
    print(f'b  = {b.item():.6f}')
    print(f'x1       = {x1.tolist()}')
    print(f'x2       = {x2.tolist()}')
    print(f'y        = {y.tolist()}')
    print(f'y_pred   = {[round(v.item(),4) for v in y_pred_final]}')
    print(f'逐样本损失 = {[round(v.item(),4) for v in loss_per_sample_final]}')
    print(f'平均损失   = {loss_final.item():.6f}')
    print(f'w1 shape = {tuple(w1.shape)},  w2 shape = {tuple(w2.shape)},  b shape = {tuple(b.shape)}')
    print(f'y_pred shape = {tuple(y_pred_final.shape)},  loss shape = {tuple(loss_final.shape)}')

x1: tensor([-2., -1.,  0.,  1.,  2.,  3.]) shape: (6,)
x2: tensor([ 1.,  0.,  2., -1.,  3.,  1.]) shape: (6,)
y:  tensor([-6., -1., -5.,  6., -4.,  4.]) shape: (6,)

===== 最终结果 =====
w1 = 1.999007
w2 = -3.001009
b  = 0.999367
x1       = [-2.0, -1.0, 0.0, 1.0, 2.0, 3.0]
x2       = [1.0, 0.0, 2.0, -1.0, 3.0, 1.0]
y        = [-6.0, -1.0, -5.0, 6.0, -4.0, 4.0]
y_pred   = [-5.9997, -0.9996, -5.0026, 5.9994, -4.0056, 3.9954]
逐样本损失 = [0.0003, 0.0004, 0.0026, 0.0006, 0.0056, 0.0046]
平均损失   = 0.002373
w1 shape = (),  w2 shape = (),  b shape = ()
y_pred shape = (6,),  loss shape = ()


## 我的观察与实验

1. 现在一次预测需要几个输入张量？x1 和 x2 与样本数 6 分别是什么关系？（10分）
2. 对照上周：预测公式、参数列表和损失计算分别改了哪里？优化器的基本步骤变了吗？（10分）
3. 记录最终参数、预测和平均损失，说明如何判断训练有效。（20分）
4. 为什么这次 w1=w2=b=0 仍然可以开始学习？（10分）

待填写。


In [ ]:
要两个输入张量，X1 X2 是shape(6,)6是长度
公式不变，但自动变成批量运算，数列不变，计算增加了.mean(),优化器不变
1.平均损失趋近于零，2预测值与真实值高度吻合，3学到的参数符合数据的内在规律
损失函数关于参数的梯度在零点不为零